# Free lens parameters with a pixelized source

This tutorial joins the nonlinear parameter framework to semi-linear source inversion. At every trial lens, Akira rebuilds the lensing operator and solves all source pixels exactly. The optimizer therefore sees only the nonlinear lens parameters.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from akira import (
    CartesianGrid, EllipticalPowerLaw, ExternalShear, FreeParameter,
    LensPlane, PixelizedLensModel, ProfileModel, SourceGrid, UniformPrior,
    fit_pixelized_model, gaussian_psf, lensing_operator, simulate_imaging,
)

## Simulate data from a pixelized source

The truth uses a 100-pixel source and an elliptical power-law lens. The reconstruction is given the correct fixed ellipticity, angle, and shear, but it must infer the Einstein radius and 3D density slope.

In [ ]:
image_grid = CartesianGrid((30, 30), pixel_scale=0.08)
source_grid = SourceGrid((10, 10), pixel_scale=0.06, centre=(0.03, -0.02))
true_mass = EllipticalPowerLaw(0.72, slope=2.08, q=0.78, phi=0.25)
true_shear = ExternalShear(0.025, phi=0.65)
true_lens = LensPlane((true_mass, true_shear))
coordinates = source_grid.coordinates()
x, y = coordinates[..., 0], coordinates[..., 1]
true_source = (
    np.exp(-0.5*((x-0.06)**2/0.07**2 + (y+0.03)**2/0.045**2))
    + 0.35*np.exp(-0.5*((x+0.08)**2/0.035**2 + (y-0.06)**2/0.05**2))
)
unblurred = (
    lensing_operator(image_grid, source_grid, true_lens) @ true_source.reshape(-1)
).reshape(image_grid.shape)
observation = simulate_imaging(
    unblurred, gaussian_psf(0.7, size=5), noise_sigma=0.008,
    pixel_scale=image_grid.pixel_scale, seed=7,
)

## Declare the nonlinear model

Only two lens fields contain `FreeParameter`. The 100 source brightnesses do not appear in this nonlinear vector: they are eliminated by the linear solve. Zeroth-order regularization is used because its positive-definite matrix defines a proper Gaussian source prior and therefore a valid marginal evidence.

In [ ]:
model = PixelizedLensModel(
    lens=(
        ProfileModel(
            EllipticalPowerLaw,
            einstein_radius=FreeParameter(0.66, UniformPrior(0.55, 0.90)),
            slope=FreeParameter(1.95, UniformPrior(1.70, 2.40)),
            centre=(0.0, 0.0), q=0.78, phi=0.25,
        ),
        ProfileModel(ExternalShear, strength=0.025, phi=0.65),
    ),
    source_grid=source_grid,
    regularization_kind='zeroth',
    regularization_strength=30.0,
)
print('nonlinear parameter vector:')
for name, value in zip(model.parameter_names, model.initial_vector):
    print(f'  {name}: {float(value):.4f}')
print(f'linear source parameters solved internally: {source_grid.size}')

## Optimize the source-marginalized evidence

For each proposed $(\theta_E,\gamma)$, the fitter performs

1. construct $F(\theta_E,\gamma)$ including lensing and the PSF;
2. solve $s=(F^TC^{-1}F+\lambda H)^{-1}F^TC^{-1}d$;
3. evaluate the Gaussian integral over every possible source vector;
4. add the nonlinear lens priors;
5. differentiate the complete calculation with JAX and update the lens.

In [ ]:
result = fit_pixelized_model(model, observation, image_grid, objective='evidence')
print(f'optimizer success: {result.success}; evaluations: {result.evaluations}')
print(f'Einstein radius: true={true_mass.einstein_radius:.5f}, '      f'fitted={float(result.parameter_vector[0]):.5f}')
print(f'3D slope:       true={true_mass.slope:.5f}, '      f'fitted={float(result.parameter_vector[1]):.5f}')
print(f'log source evidence: {result.log_source_evidence:.2f}')
print(f'image chi-squared:   {result.reconstruction.chi_squared:.2f}')

The evidence includes determinant terms that penalize source flexibility, not just image residuals. The alternatives `objective='penalized'` and `objective='profile_likelihood'` are available as diagnostics. Those alternatives also permit gradient or curvature regularization, whose current null modes make their absolute Gaussian evidence undefined.

In [ ]:
reconstruction = result.reconstruction
source_extent = (-0.27, 0.33, -0.32, 0.28)
fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
vmin = min(float(np.min(true_source)), float(np.min(reconstruction.source)))
vmax = max(float(np.max(true_source)), float(np.max(reconstruction.source)))
for axis, values, title in zip(
    axes, (true_source, reconstruction.source), ('true source', 'inferred source')
):
    shown = axis.imshow(values, origin='lower', extent=source_extent, cmap='magma',
                        vmin=vmin, vmax=vmax)
    axis.set_title(title); axis.set_xlabel('source x [arcsec]')
axes[0].set_ylabel('source y [arcsec]')
fig.colorbar(shown, ax=axes, fraction=0.04); plt.show()

In [ ]:
image_extent = (-1.2, 1.2, -1.2, 1.2)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
panels = (
    (observation.data, 'data', 'magma'),
    (reconstruction.model_image, 'best model', 'magma'),
    (reconstruction.residuals, 'residual / noise', 'coolwarm'),
)
for axis, (values, title, cmap) in zip(axes, panels):
    shown = axis.imshow(values, origin='lower', extent=image_extent, cmap=cmap)
    axis.set_title(title); axis.set_xlabel('image x [arcsec]')
    fig.colorbar(shown, ax=axis, fraction=0.046)
axes[0].set_ylabel('image y [arcsec]')
fig.tight_layout(); plt.show()

This small dense problem is a correctness demonstration. Each nonlinear evaluation rebuilds and differentiates through the full dense operator, source solve, and matrix determinants. Larger scientific applications will need the planned sparse or matrix-free implementation and more careful treatment of source-grid coverage and regularization hyperparameters.